# 02 – Data preparation: Secondary Mushroom

**Worked on by:** Andreas Schellekens (cleaning steps, preprocessing pipeline, sanity checks)

> **GenAI disclosure:** the first version of this notebook was drafted with the help of Claude Code (AI assistant), based on the decisions we made in `01_eda.ipynb`. All steps were reviewed by the team; we must be able to explain every cell ourselves.

## 1. Setup & loading

Imports, file paths and constants. `RANDOM_STATE` is fixed so the train / validation / test split is the same every time the notebook runs (and the same for every teammate).
`LOKY_MAX_CPU_COUNT` silences a harmless joblib warning on Windows (joblib cannot count the physical CPU cores there); it must be set before scikit-learn is imported.

Every set gets its own folder in `Data/` (`Data/train/`, `Data/validation/`, `Data/test/`), with a *cleaned* file (readable values, for PyCaret and the deployable pipelines) and a *prepared* file (model-ready numbers). `split_path` builds these file names, so they are written in one place only.

In [1]:
import os
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "4")

from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

DATA_DIR = Path("Data")
MODEL_DIR = Path("models")
RAW_PATH = DATA_DIR / "mushroom_project_dataset.csv"
PREPROCESSOR_PATH = MODEL_DIR / "mushroom_preprocessor.joblib"

RANDOM_STATE = 42
SPLITS = ["train", "validation", "test"]
SPLIT_SHARES = {"train": 0.70, "validation": 0.15, "test": 0.15}


def split_path(split, kind):
    # kind is "cleaned" or "prepared", e.g. Data/validation/mushroom_prepared_validation.csv
    return DATA_DIR / split / f"mushroom_{kind}_{split}.csv"

We read the raw CSV. It is only read, never overwritten, so the original stays available.
The `assert` checks that the file is the one we explored in the EDA (5000 rows, 13 columns). If the lecturer ever hands out a new version, the notebook stops here instead of silently producing something different.

In [2]:
df_raw = pd.read_csv(RAW_PATH)
assert df_raw.shape == (5000, 13), f"Unexpected shape {df_raw.shape} – is this the dataset from the EDA?"
print(f"Raw data: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")

Raw data: 5,000 rows × 13 columns


## 2. Column names & readable labels
*(EDA section 2 – decision table steps 1 and 2)*

- Column names: `cap-diameter` → `cap_diameter` (snake_case works as a Python attribute and as a JSON key in the API).
- The categorical values are one-letter codes. We translate them with the UCI dictionary, so later outputs (feature importances, the web interface) are readable.
- The code `f` means **"none"** for `gill_color`, `stem_surface` and `ring_type` (no gills / no stem surface / no ring). That is real information, not a missing value, so it becomes the label `none` – different from `NaN`.

The noise columns are not translated; we drop them in the next step anyway.

In [3]:
COLOR = {"n": "brown", "b": "buff", "g": "gray", "r": "green", "p": "pink", "u": "purple",
         "e": "red", "w": "white", "y": "yellow", "l": "blue", "o": "orange", "k": "black", "f": "none"}

CODE_MAPS = {
    "class":             {"e": "edible", "p": "poisonous"},
    "cap_shape":         {"b": "bell", "c": "conical", "x": "convex", "f": "flat",
                          "s": "sunken", "p": "spherical", "o": "others"},
    "gill_color":        COLOR,
    "spore_print_color": COLOR,
    "stem_surface":      {"i": "fibrous", "g": "grooves", "y": "scaly", "s": "smooth", "h": "shiny",
                          "l": "leathery", "k": "silky", "t": "sticky", "w": "wrinkled", "e": "fleshy", "f": "none"},
    "ring_type":         {"c": "cobwebby", "e": "evanescent", "r": "flaring", "g": "grooved", "l": "large",
                          "p": "pendant", "s": "sheathing", "z": "zone", "y": "scaly", "m": "movable", "f": "none"},
    "habitat":           {"g": "grasses", "l": "leaves", "m": "meadows", "p": "paths", "h": "heaths",
                          "u": "urban", "w": "waste", "d": "woods"},
    "season":            {"s": "spring", "u": "summer", "a": "autumn", "w": "winter"},
}

We work on a copy (`df`) so `df_raw` stays untouched for comparison.
Before translating, we check that every code in the data exists in its dictionary: `.map()` turns an unknown code into `NaN` without any warning, which would silently create extra missing values.

In [4]:
df = df_raw.copy()
df.columns = df.columns.str.strip().str.lower().str.replace("-", "_")

for col, mapping in CODE_MAPS.items():
    unknown = set(df[col].dropna().unique()) - set(mapping)
    assert not unknown, f"{col} has unmapped codes: {unknown}"
    df[col] = df[col].map(mapping)

# Translating must not create or remove missing values
assert df.isna().sum().sum() == df_raw.isna().sum().sum()
df.columns.tolist()

['cap_diameter',
 'stem_height',
 'stem_width',
 'spore_print_color',
 'gill_color',
 'habitat',
 'season',
 'ring_type',
 'cap_shape',
 'stem_surface',
 'jumbled_noise_0',
 'jumbled_noise_1',
 'class']

## 3. Drop the noise columns
*(EDA section 4 – step 3)*

`jumbled_noise_0` and `jumbled_noise_1` are row-shuffled copies of `cap_shape`: identical value counts, agreement with `cap_shape` at chance level, and no association with the class after a Bonferroni correction. Keeping them only gives a model the opportunity to fit noise. Section 9 below checks that dropping them indeed helps.

In [5]:
NOISE_COLS = ["jumbled_noise_0", "jumbled_noise_1"]
df = df.drop(columns=NOISE_COLS)

NUM_COLS = ["cap_diameter", "stem_height", "stem_width"]
CAT_COLS = ["spore_print_color", "gill_color", "habitat", "season", "ring_type", "cap_shape", "stem_surface"]
TARGET = "class"
assert set(df.columns) == set(NUM_COLS + CAT_COLS + [TARGET])

## 4. Missing values in categoricals
*(EDA section 5 – steps 4 and 5)*

- We **keep every row**. Only 12 of the 5000 rows are complete, so `dropna()` would throw away almost the whole dataset.
- For categoricals, `NaN` becomes the explicit category `"missing"`. For `stem_surface` and `spore_print_color` the fact that the value is missing is related to the class (EDA 5.2), so the model must be able to *see* it. For the other categoricals it is harmless: the model can simply learn that `"missing"` means nothing special.
- `spore_print_color` is 95% missing. Imputing it would mean inventing almost the whole column, so `"missing"` is the only honest option. Whether to drop the column entirely is an open question for the model notebooks (it is easy to drop there).
- Numerical `NaN` are **not** filled here: the median must be learned on the training set only (section 7).

In [6]:
df[CAT_COLS] = df[CAT_COLS].fillna("missing")
df[CAT_COLS].isna().sum().sum(), df[NUM_COLS].isna().sum().to_dict()

(0, {'cap_diameter': 2032, 'stem_height': 1013, 'stem_width': 395})

## 5. Stem consistency & the `has_stem` feature
*(EDA section 6.1 – step 7)*

In the EDA we found that `stem_height = 0` and `stem_width = 0` always occur together: these are mushrooms **without a stem**, and 92% of them are poisonous. Three extra facts, checked while building this notebook:
- `stem_surface = none` occurs **only** on stemless mushrooms (56 of 56 rows) – it is a third way to recognise them.
- Some stemless mushrooms miss one of the two measures (11 rows have width 0 but height missing, 3 the other way round).
- For 83 rows both measures are missing, but for some of those `stem_surface` still tells us whether there is a stem.

Why this matters: median imputation (section 7) would give a stemless mushroom a height of ~6 cm, i.e. turn it into a "normal" mushroom and erase the strongest signal in the numericals. So before imputing, we use the information we *do* have:

1. `has_stem = 0` if any of the three sources says "no stem"; `has_stem = 1` if any says there is a stem (a measure > 0 or a real stem surface); otherwise unknown (`NaN`, imputed in section 7).
2. For stemless mushrooms, set a missing stem measure to 0 and a missing `stem_surface` to `none`. These are logical consequences, not guesses.

First we check that the three sources never contradict each other; if they did, one of them would be wrong and we couldn't trust this rule.

In [7]:
height, width, surface = df["stem_height"], df["stem_width"], df["stem_surface"]
says_no_stem = (height == 0) | (width == 0) | (surface == "none")
says_stem = (height > 0) | (width > 0) | ~surface.isin(["none", "missing"])

assert not (says_no_stem & says_stem).any(), "Contradicting stem information"
print(f"No stem: {says_no_stem.sum()} | stem: {says_stem.sum()} | unknown: {(~says_no_stem & ~says_stem).sum()}")

No stem: 60 | stem: 4887 | unknown: 53


Now we apply the rule. `has_stem` uses pandas' nullable integer type `Int64`, so it can be 0, 1 or missing (written as an empty cell in the CSV).
We print how many values the logical fill repaired.

In [8]:
filled = {
    "stem_height": (says_no_stem & height.isna()).sum(),
    "stem_width": (says_no_stem & width.isna()).sum(),
    "stem_surface": (says_no_stem & (surface == "missing")).sum(),
}
df.loc[says_no_stem, ["stem_height", "stem_width"]] = 0.0
df.loc[says_no_stem, "stem_surface"] = "none"
df["has_stem"] = pd.Series(pd.NA, index=df.index, dtype="Int64")  # start as unknown
df.loc[says_stem, "has_stem"] = 1
df.loc[says_no_stem, "has_stem"] = 0

print("Values filled by the stem rule:", filled)
print(pd.crosstab(df["has_stem"].astype("object").fillna("unknown"), df[TARGET], margins=True))

Values filled by the stem rule: {'stem_height': 11, 'stem_width': 3, 'stem_surface': 4}
class     edible  poisonous   All
has_stem                         
0              4         56    60
1           3070       1817  4887
unknown       33         20    53
All         3107       1893  5000


**Outliers** *(EDA 6.3 – step 7)*: we deliberately do **nothing**. The large values are biologically plausible (caps of 30–60 cm exist), removing them would cost 3.6–6.8% of the known values per column (EDA 6.3) of an already small dataset, and there are no impossible values. We only verify that last claim, so the notebook fails if a future version of the data contains negative sizes.

In [9]:
assert (df[NUM_COLS].dropna() >= 0).all().all(), "Negative sizes found – physically impossible"

**Ordered categoricals** *(EDA 7.3)*: none of the categoricals has a natural order (`season` is cyclic), so they stay nominal and will be one-hot encoded – no ordinal encoding.

## 6. Train / validation / test split & saving the cleaned dataset
*(EDA section 3 – step 10)*

We split **once, here**, into three sets, and store which row belongs to which set. Every model notebook (baseline, PyCaret, tuned models, SageMaker) uses the **same** three sets, so their scores can be compared fairly in `07_model_comparison.ipynb`.

| Set | Share | Rows | Used for |
|---|---|---|---|
| train | 70% | 3500 | Fitting the models. Hyperparameter search with 5-fold cross-validation *inside* this set. |
| validation | 15% | 750 | Never used to fit a model. Choosing the decision threshold, checking that a tuned model really beats the untuned one, and choosing between models (`07`). |
| test | 15% | 750 | The final evaluation, once per model, after every choice has been made. |

**Why a separate validation set?** With only a train and a test set, every choice was made with cross-validation on the training rows. The decision threshold, for example, was chosen on out-of-fold predictions from the same folds that had just been used to pick the hyperparameters, which makes it slightly optimistic. And choosing *the best of several models* on the test set would quietly turn the test set into a selection set: its score would no longer be an honest estimate for new mushrooms. The validation set gives those choices their own data, so the test set is only used to *report*.

**Why still cross-validation inside train for the hyperparameter search?** 750 rows are too few to pick the best of 40 candidates: the ROC-AUC on 750 rows has a standard error of about 0.015, larger than the differences between good candidates, so the "winner" would partly be the one that fits these 750 rows by luck. Cross-validation averages over 5 folds of 3500 rows. The validation set then checks the winner once.

**The cost:** 500 fewer training rows than an 80/20 split, and a test set of 750 instead of 1000 rows (standard error of the test AUC about 0.015 instead of 0.013). With 5000 rows this is affordable.

How: first the test set is split off, then the rest is split into train and validation. Both steps are **stratified**, so all three sets keep the 62/38 class ratio, and both use `RANDOM_STATE`. The sizes are given as row counts (750), so rounding cannot make a set one row bigger or smaller. The sets must stay separate until their moment: also experiments like removing suspected flipped labels (EDA section 9) only use the training set.

In [10]:
n_test = round(SPLIT_SHARES["test"] * len(df))
n_validation = round(SPLIT_SHARES["validation"] * len(df))

rest_idx, test_idx = train_test_split(df.index, test_size=n_test, stratify=df[TARGET], random_state=RANDOM_STATE)
train_idx, validation_idx = train_test_split(rest_idx, test_size=n_validation, stratify=df.loc[rest_idx, TARGET],
                                             random_state=RANDOM_STATE)
df["split"] = "train"
df.loc[validation_idx, "split"] = "validation"
df.loc[test_idx, "split"] = "test"

pd.crosstab(df["split"], df[TARGET], normalize="index").round(3).assign(rows=df["split"].value_counts()).loc[SPLITS]

class,edible,poisonous,rows
split,,,
train,0.621,0.379,3500
validation,0.621,0.379,750
test,0.621,0.379,750


**Duplicates.** The EDA found no duplicate rows, but that was *with* the noise columns (whose shuffled values made every row unique) and *before* the stem rule filled in zeros. We check again now, per set, because identical rows in train and test (or validation) would give a model a few answers "for free".

In [11]:
features = NUM_COLS + ["has_stem"] + CAT_COLS
dup_rows = df[df.duplicated(subset=features, keep=False)]
n_groups = dup_rows.groupby(features, dropna=False).ngroups
print(f"Rows that have an identical twin (on all features): {len(dup_rows)} in {n_groups} groups")
print("Groups with conflicting labels:", (dup_rows.groupby(features, dropna=False)[TARGET].nunique() > 1).sum())
print("Groups that span more than one set:", (dup_rows.groupby(features, dropna=False)["split"].nunique() > 1).sum())
dup_rows.sort_values(features)[features + [TARGET, "split"]]

Rows that have an identical twin (on all features): 16 in 7 groups
Groups with conflicting labels: 0
Groups that span more than one set: 5


,cap_diameter,stem_height,stem_width,has_stem,spore_print_color,gill_color,habitat,season,ring_type,cap_shape,stem_surface,class,split
273,4.91,0.0,0.0,0,brown,white,woods,missing,none,others,none,poisonous,train
621,4.91,0.0,0.0,0,brown,white,woods,missing,none,others,none,poisonous,validation
1969,NaN,0.0,0.0,0,brown,white,woods,missing,none,others,none,poisonous,train
2473,NaN,0.0,0.0,0,brown,white,woods,missing,none,others,none,poisonous,validation
580,NaN,0.0,0.0,0,missing,missing,woods,autumn,none,others,none,poisonous,train
4082,NaN,0.0,0.0,0,missing,missing,woods,autumn,none,others,none,poisonous,test
232,NaN,0.0,0.0,0,missing,none,woods,winter,none,others,none,poisonous,train
3164,NaN,0.0,0.0,0,missing,none,woods,winter,none,others,none,poisonous,train
3720,NaN,0.0,0.0,0,missing,none,woods,winter,none,others,none,poisonous,validation
4244,NaN,0.0,0.0,0,missing,none,woods,winter,none,others,none,poisonous,validation


**Decision: keep the duplicates.** They are not copied records. 12 of the 16 rows are stemless, poisonous mushrooms of what looks like one species (`cap_shape = others`, woods, no stem); the other 4 are edible mushrooms with no size measurements at all. In both cases so many cells were blanked that different mushrooms have become indistinguishable. The dataset has many mushrooms per species, so this is expected. None of them have conflicting labels. Dropping them would thin out exactly the small, high-risk stemless group. 5 of the 7 groups have members in more than one set, but that concerns only a handful of rows in the validation and test sets of 750 each (a few tenths of a percent on a score). In real use, the API would also receive such "look-alike" input.

We save the cleaned dataset, **one file per set, in the folder of that set** (`Data/train/mushroom_cleaned_train.csv`, and the same for `validation` and `test`). Numerical `NaN` are written as empty cells; categoricals already use `"missing"`. The original row number is saved as `row_id`, the same key as in the prepared files, so every row can be traced back to the raw file. The raw file is not touched.

These files are what PyCaret / AutoML, the gradient boosting model with native missing values and the deployable pipelines get: they do their own imputation and encoding, or use our fitted preprocessor.

In [12]:
CLEANED_COLS = NUM_COLS + ["has_stem"] + CAT_COLS + [TARGET]
for split in SPLITS:
    path = split_path(split, "cleaned")
    path.parent.mkdir(parents=True, exist_ok=True)
    df.loc[df["split"] == split, CLEANED_COLS].to_csv(path, index_label="row_id")
    print(f"Saved {path}  ({(df['split'] == split).sum():,} rows, {len(CLEANED_COLS)} columns + row_id)")
df[CLEANED_COLS].head()

Saved Data\train\mushroom_cleaned_train.csv  (3,500 rows, 12 columns + row_id)
Saved Data\validation\mushroom_cleaned_validation.csv  (750 rows, 12 columns + row_id)
Saved Data\test\mushroom_cleaned_test.csv  (750 rows, 12 columns + row_id)


,cap_diameter,stem_height,stem_width,has_stem,spore_print_color,gill_color,habitat,season,ring_type,cap_shape,stem_surface,class
0,NaN,6.42,13.51,1,missing,missing,woods,summer,grooved,convex,missing,poisonous
1,6.99,7.68,18.76,1,missing,white,woods,autumn,none,flat,smooth,edible
2,5.78,7.67,8.21,1,missing,green,woods,summer,none,conical,missing,poisonous
3,6.34,8.34,11.06,1,missing,missing,woods,autumn,zone,convex,missing,poisonous
4,NaN,5.67,12.78,1,missing,brown,woods,missing,none,convex,missing,edible


## 7. Preprocessing pipeline: imputation, normalisation, encoding
*(EDA sections 5.3, 6.2 and 7.1 – steps 6, 8 and 9)*

Everything that *learns from the data* goes into one scikit-learn `ColumnTransformer`, fitted on the training rows only:

| Columns | Steps | Why |
|---|---|---|
| `cap_diameter`, `stem_height`, `stem_width` | median imputation → `log1p` → standardisation | Missing at random (EDA 5.2), median is robust to the skew. `log1p` = log(1+x): removes the strong right skew (skewness 4.2 → 0.1) and still works for the zeros of stemless mushrooms. Standardising (mean 0, std 1) puts all features on the same scale, needed for logistic regression, KNN, SVM and neural networks. |
| `has_stem` | most-frequent imputation | Only the ~50 rows without any stem information are imputed; almost all mushrooms have a stem. |
| 7 categoricals | one-hot encoding, categories with < 10 training rows grouped into one "infrequent" column | No order (7.3). Very rare categories are unreliable (7.1) and give almost-empty columns. |

**Why one pipeline for all models?** `log1p` and standardisation are *monotonic*: they don't change the order of the values. Tree models (random forest, gradient boosting) only look at the order, so for them these steps change nothing, while linear and distance-based models need them. So one prepared dataset serves every model.

**Paths not taken:**
- *Rare-category threshold 30* (as first suggested in the EDA): it would also merge small but almost pure categories such as `stem_surface = shiny` (n=27, 93% poisonous) and `habitat = paths` (n=22, 82% poisonous) into one mixed group, which destroys real signal. With 10 we only merge categories that are too small to trust (e.g. spore colours purple/green, n=3). Section 9 compares both.
- *KNN / iterative imputation* of `cap_diameter` using its correlation with `stem_width` (ρ = 0.85): a promising experiment, but not adopted without evidence. Section 9.2 below tests it with cross-validation on the training set: the gain is within the noise. Median is the simple, robust default.
- *Outlier clipping*: not needed – after `log1p` the extremes are no longer extreme.

`handle_unknown="infrequent_if_exist"` has one more advantage for deployment: if the API receives a category the model never saw, it is treated as "infrequent" instead of crashing.

In [13]:
numeric_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("scale", StandardScaler()),
])

preprocessor = ColumnTransformer(
    [
        ("num", numeric_pipeline, NUM_COLS),
        ("stem", SimpleImputer(strategy="most_frequent"), ["has_stem"]),
        ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=10, sparse_output=False), CAT_COLS),
    ],
    verbose_feature_names_out=True,
)

We fit the preprocessor on the **training rows only**, then use it to transform all three sets. The validation and test rows are transformed with the medians, scaling and categories learned on train, exactly like a new mushroom in the API will be.
`has_stem` is converted to a plain float first, because scikit-learn's imputer expects `NaN` rather than pandas' `pd.NA`.
The target becomes `is_poisonous` (1 = poisonous): poisonous is the class we care about most (recall!), so we make it the positive class.

In [14]:
def to_model_input(frame):
    # Select the feature columns in the order the preprocessor expects
    X = frame[NUM_COLS + ["has_stem"] + CAT_COLS].copy()
    X["has_stem"] = X["has_stem"].astype("float")
    return X


X, y = {}, {}
for split in SPLITS:
    rows = df[df["split"] == split]
    X[split] = to_model_input(rows)
    y[split] = (rows[TARGET] == "poisonous").astype(int).rename("is_poisonous")
X_train, y_train = X["train"], y["train"]  # used again in section 9

preprocessor.fit(X_train)
feature_names = preprocessor.get_feature_names_out()

prepared = {split: pd.DataFrame(preprocessor.transform(X[split]), columns=feature_names, index=X[split].index).join(y[split])
            for split in SPLITS}
print(f"{len(feature_names)} features after encoding")

64 features after encoding


Which categories were grouped as infrequent? This makes the effect of `min_frequency=10` visible.

In [15]:
encoder = preprocessor.named_transformers_["cat"]
{col: list(cats) for col, cats in zip(CAT_COLS, encoder.infrequent_categories_) if cats is not None}

{'spore_print_color': ['green', 'purple'], 'habitat': ['urban']}

We save the prepared sets, again one per folder (`Data/train/mushroom_prepared_train.csv`, ...), and the fitted preprocessor. The original index is kept as `row_id`, so every prepared row can be traced back to the raw file (useful for error analysis later).
The preprocessor is saved with `joblib` (the scikit-learn recommended format; it is only a few KB, far below GitHub's 100 MB limit).

In [16]:
MODEL_DIR.mkdir(exist_ok=True)
for split in SPLITS:
    prepared[split].to_csv(split_path(split, "prepared"), index_label="row_id")
joblib.dump(preprocessor, PREPROCESSOR_PATH)

for path in [split_path(split, "prepared") for split in SPLITS] + [PREPROCESSOR_PATH]:
    print(f"Saved {path}  ({path.stat().st_size / 1024:.0f} KB)")
prepared["train"].head()

Saved Data\train\mushroom_prepared_train.csv  (1065 KB)
Saved Data\validation\mushroom_prepared_validation.csv  (229 KB)
Saved Data\test\mushroom_prepared_test.csv  (229 KB)
Saved models\mushroom_preprocessor.joblib  (7 KB)


,num__cap_diameter,num__stem_height,num__stem_width,stem__has_stem,cat__spore_print_color_black,cat__spore_print_color_brown,cat__spore_print_color_gray,cat__spore_print_color_missing,cat__spore_print_color_pink,cat__spore_print_color_white,...,cat__stem_surface_fibrous,cat__stem_surface_grooves,cat__stem_surface_missing,cat__stem_surface_none,cat__stem_surface_scaly,cat__stem_surface_shiny,cat__stem_surface_silky,cat__stem_surface_smooth,cat__stem_surface_sticky,is_poisonous
1,0.304613,0.546541,0.812209,1.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0
2,-0.067528,0.543554,-0.237722,1.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1
5,0.049715,1.370525,0.135368,1.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0
6,0.049715,0.582114,-0.222869,1.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0
9,0.049715,-0.014487,0.737846,1.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0


## 8. Sanity checks

Automatic checks that the prepared data is what we intended. If any of them fails, the notebook stops.
1. No missing values left, and every row is kept exactly once (3500 + 750 + 750 = 5000, no row in two sets).
2. The three sets have exactly the same columns.
3. The class ratio is the same in all three sets (stratification worked).
4. The numerical features have mean ≈ 0 and std ≈ 1 **on the training set**. (On validation and test they will be close but not exact – that is expected, since the scaler never saw those rows.)
5. Reading the saved files back gives the same data, and the saved preprocessor turns the saved *cleaned* test file into exactly the saved *prepared* test file (so the API, which starts from cleaned values, will transform input exactly as during training).

In [17]:
all_rows = pd.concat([prepared[split] for split in SPLITS])
assert all_rows.notna().all().all()
assert len(all_rows) == len(df_raw) and all_rows.index.is_unique
for split in SPLITS:
    assert list(prepared[split].columns) == list(prepared["train"].columns)
    assert abs(y[split].mean() - y_train.mean()) < 0.01

num_features = [f"num__{c}" for c in NUM_COLS]
print("Numerical features after preprocessing:")
print(pd.concat({(split, stat): getattr(prepared[split][num_features], stat)() for split in SPLITS for stat in ["mean", "std"]},
                axis=1).round(3))

for split in SPLITS:
    reloaded = pd.read_csv(split_path(split, "prepared"), index_col="row_id")
    assert reloaded.index.equals(prepared[split].index) and np.allclose(reloaded.values, prepared[split].values)
cleaned_test = pd.read_csv(split_path("test", "cleaned"), index_col="row_id")
assert np.allclose(joblib.load(PREPROCESSOR_PATH).transform(to_model_input(cleaned_test)), prepared["test"][feature_names].values)
print("\nAll sanity checks passed.")

Numerical features after preprocessing:
                  train      validation          test       
                   mean  std       mean    std   mean    std
num__cap_diameter   0.0  1.0      0.016  1.023 -0.027  0.976
num__stem_height    0.0  1.0     -0.030  1.089  0.014  0.943
num__stem_width    -0.0  1.0     -0.078  1.066 -0.049  1.045

All sanity checks passed.


## 9. Does the preparation actually help?

### 9.1 The cleaning steps

The assignment asks to check that our changes actually improve the data. We compare a few versions of the preparation with the same model and the same **5-fold cross-validation on the training set only**. The test set stays untouched, and so does the validation set: it is kept for the model decisions in the model notebooks (threshold, choice of model). If we also used it for every small preparation experiment, it would slowly become part of the tuning as well.

- Model: a random forest with default-ish settings. It is not our final model; it's a fixed "measuring instrument". We use a non-linear model because the EDA showed that is what this data needs (a quick check with logistic regression showed only differences of ±0.5 percentage points: a linear model is too weak here to benefit much from any of these steps).
- Metrics: accuracy, **recall of the poisonous class** (the costly error, EDA section 3) and ROC-AUC.

Versions:
1. `noise kept`: our full preparation, but with the two `jumbled_noise` columns added back.
2. `no stem fix`: noise dropped, but without the stem rule (stemless mushrooms with a missing measure get the median, no `has_stem`).
3. `rare threshold 30`: our preparation with `min_frequency=30` instead of 10.
4. `final`: the preparation of this notebook.

In [18]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate


def cv_scores(preproc, X, y):
    model = Pipeline([("prep", preproc),
                      ("rf", RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1))])
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    scores = cross_validate(model, X, y, cv=cv, scoring=["accuracy", "recall", "roc_auc"])
    return {m: scores[f"test_{m}"].mean() for m in ["accuracy", "recall", "roc_auc"]}


def variant(num_cols, stem_cols, cat_cols, min_frequency=10):
    # Same structure as the final preprocessor, with configurable columns
    steps = [("num", clone(numeric_pipeline), num_cols),
             ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=min_frequency, sparse_output=False), cat_cols)]
    if stem_cols:
        steps.append(("stem", SimpleImputer(strategy="most_frequent"), stem_cols))
    return ColumnTransformer(steps)


# 1. Noise columns back in (taken from the raw data, same rows as the training set; NaN -> "missing")
X_noise = X_train.join(df_raw.loc[X_train.index, NOISE_COLS].fillna("missing"))
# 2. Stem measures as they were before the stem rule (translated and noise-free, but not repaired)
X_no_fix = X_train.copy()
X_no_fix[["stem_height", "stem_width"]] = df_raw.loc[X_train.index, ["stem-height", "stem-width"]].values
X_no_fix["stem_surface"] = df_raw.loc[X_train.index, "stem-surface"].map(CODE_MAPS["stem_surface"]).fillna("missing")

results = {
    "noise kept":        cv_scores(variant(NUM_COLS, ["has_stem"], CAT_COLS + NOISE_COLS), X_noise, y_train),
    "no stem fix":       cv_scores(variant(NUM_COLS, [], CAT_COLS), X_no_fix, y_train),
    "rare threshold 30": cv_scores(variant(NUM_COLS, ["has_stem"], CAT_COLS, min_frequency=30), X_train, y_train),
    "final":             cv_scores(preprocessor, X_train, y_train),
}
pd.DataFrame(results).T.round(3)

,accuracy,recall,roc_auc
noise kept,0.759,0.450,0.819
no stem fix,0.773,0.497,0.830
rare threshold 30,0.773,0.497,0.829
final,0.773,0.503,0.831


**Interpretation** *(numbers from our run; they can move by a few tenths of a percent between library versions)*
- **Dropping the noise columns clearly helps**: with them the forest loses about 1.5 points of accuracy, 5 points of poisonous recall and 0.012 AUC. A random forest picks features at random for its splits, so useless columns regularly get chosen and the model fits noise. This confirms the EDA conclusion with a model instead of a statistical test.
- **The stem rule changes almost nothing in the scores** (same accuracy, +0.6 points recall, +0.001 AUC). It only changes about 20 rows and adds one feature, so a small effect is what we'd expect. We keep it because it is logically correct (a stemless mushroom should not get an invented stem) and it concerns the high-risk group (92% poisonous), not because of the score.
- **Threshold 10 vs. 30** for rare categories: 30 is slightly worse (−0.6 points recall, −0.002 AUC), in line with the reasoning in section 7.
- Apart from the noise columns, the differences are smaller than the variation between CV folds. So we read this as "these steps don't hurt and probably help", not as a proven improvement. The real comparison happens in the model notebooks.
- Poisonous recall is only around 50% at the default 0.5 threshold. That is a modelling problem, not a data problem: in the model notebooks we should lower the decision threshold and/or use class weights, since missing a poisonous mushroom is the costly error.

### 9.2 Open questions: `spore_print_color`, imputation, ratio features

The EDA left three questions open. We answer them with the same measuring instrument, each as one change to `final`:

5. `drop spore_print_color`: the column is 95% missing. Is the 5% that is left worth a column?
6. `iterative imputation`: `cap_diameter` is 41% missing and strongly correlated with `stem_width` (Spearman ρ = 0.85, EDA section 8). `IterativeImputer` predicts every missing size from the other sizes (one regression per column, repeated until the values stop changing) instead of filling in the median. It runs on the `log1p` scale, where the relation between the sizes is closer to linear.
7. `ratio features`: two extra columns, stem width ÷ stem height (slender or stocky stem) and cap diameter ÷ stem width. We compute them as differences of the `log1p` values, which avoids a division by zero for stemless mushrooms. A tree splits on one feature at a time, so it can only approximate a ratio with many small steps; giving it the ratio directly could help.
8. `iterative + ratios`: both changes together.

**A more precise comparison.** The differences we expect are small, smaller than the fold-to-fold variation in the table above. So we:
- repeat 5-fold CV 3 times with different shuffles (15 folds instead of 5), and
- compare each version **fold by fold** with `final`. Both are scored on exactly the same rows, so the variation between folds (some folds are simply harder) cancels out in the difference.

**Decision rule, fixed before looking at the results:** a change is adopted only if its mean AUC gain is more than twice its standard error. The standard error needs a correction (Nadeau & Bengio, 2003): the 15 folds reuse the same training rows, so their scores are not independent and the usual formula `std / √15` would be far too small. The corrected version is `std × √(1/15 + 1/4)`, where 1/4 is the ratio of held-out to training rows within one fold of 5-fold CV.

`IterativeImputer` is still marked *experimental* in scikit-learn, which is why it needs the extra `enable_iterative_imputer` import.

In [19]:
from sklearn.experimental import enable_iterative_imputer  # noqa: F401  (unlocks IterativeImputer)
from sklearn.impute import IterativeImputer
from sklearn.model_selection import RepeatedStratifiedKFold


def add_log_ratios(Z):
    # Z holds log1p(cap_diameter), log1p(stem_height), log1p(stem_width); a log difference is a log ratio
    Z = np.asarray(Z)
    return np.column_stack([Z, Z[:, 2] - Z[:, 1], Z[:, 0] - Z[:, 2]])


def numeric_variant(imputer, ratios=False):
    # log1p first, so the iterative imputer works on the (more linear) log scale.
    # For the median this order changes nothing: the median of log1p(x) is log1p of the median.
    steps = [("log1p", FunctionTransformer(np.log1p)), ("impute", imputer)]
    if ratios:
        steps.append(("ratios", FunctionTransformer(add_log_ratios)))
    return Pipeline(steps + [("scale", StandardScaler())])


def prep_variant(numeric=None, cat_cols=CAT_COLS):
    return ColumnTransformer([
        ("num", numeric if numeric is not None else clone(numeric_pipeline), NUM_COLS),
        ("stem", SimpleImputer(strategy="most_frequent"), ["has_stem"]),
        ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=10, sparse_output=False), cat_cols),
    ])


repeated_cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)


def fold_scores(preproc):
    model = Pipeline([("prep", preproc),
                      ("rf", RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1))])
    scores = cross_validate(model, X_train, y_train, cv=repeated_cv, scoring=["accuracy", "recall", "roc_auc"])
    return pd.DataFrame({m: scores[f"test_{m}"] for m in ["accuracy", "recall", "roc_auc"]})


# max_iter=30: with the default of 10 rounds the imputed values are still changing (it converges after about 15)
iterative = IterativeImputer(max_iter=30, random_state=RANDOM_STATE)
open_questions = {
    "drop spore_print_color": prep_variant(cat_cols=[c for c in CAT_COLS if c != "spore_print_color"]),
    "iterative imputation":   prep_variant(numeric_variant(clone(iterative))),
    "ratio features":         prep_variant(numeric_variant(SimpleImputer(strategy="median"), ratios=True)),
    "iterative + ratios":     prep_variant(numeric_variant(clone(iterative), ratios=True)),
}

final_folds = fold_scores(preprocessor)
n_folds = len(final_folds)
se_factor = np.sqrt(1 / n_folds + 1 / 4)  # Nadeau & Bengio correction

paired = {}
for name, preproc in open_questions.items():
    diff = fold_scores(preproc) - final_folds
    se = diff["roc_auc"].std() * se_factor
    paired[name] = {
        "diff_accuracy": diff["accuracy"].mean(),
        "diff_recall": diff["recall"].mean(),
        "diff_roc_auc": diff["roc_auc"].mean(),
        "se_diff_roc_auc": se,
        "gain_in_se": diff["roc_auc"].mean() / se,
        "folds_with_higher_auc": (diff["roc_auc"] > 0).sum(),
    }
print(f"final (reference): ROC-AUC {final_folds['roc_auc'].mean():.3f} over {n_folds} folds")
pd.DataFrame(paired).T.astype({"folds_with_higher_auc": int}).round(4)

final (reference): ROC-AUC 0.830 over 15 folds


,diff_accuracy,diff_recall,diff_roc_auc,se_diff_roc_auc,gain_in_se,folds_with_higher_auc
drop spore_print_color,-0.0042,-0.0078,-0.0041,0.0028,-1.4303,3
iterative imputation,0.0050,0.0141,0.0003,0.0036,0.0908,9
ratio features,0.0070,0.0287,0.0044,0.0038,1.1617,10
iterative + ratios,0.0110,0.0370,0.0050,0.0037,1.3593,11


**Interpretation** *(differences with `final`, over the same 15 folds)*
- **Keep `spore_print_color`.** Dropping it lowers the AUC in 12 of the 15 folds, by about 0.004 on average (1.4 standard errors). It does not pass our bar in the other direction either, but there is no reason to remove a column that, if anything, helps: the 5% of mushrooms that do have a spore print colour carry real signal (e.g. `brown` mostly poisonous, `gray` mostly edible; see the baseline coefficients in notebook 03), and the `"missing"` category costs nothing.
- **Iterative imputation alone adds nothing** (+0.0003 AUC, higher in only 9 of 15 folds). Predicting `cap_diameter` from `stem_width` sounds better than the median, but the forest does not get better from it.
- **The ratio features point in the right direction, but not clearly enough**: +0.004 AUC alone, +0.005 together with iterative imputation (higher in 11 of 15 folds). That is 1.2 and 1.4 standard errors, below the bar of 2 we set in advance. The extra poisonous recall (+3 to 4 points at threshold 0.5) mostly means the probabilities moved a little; recall is set with the threshold in the model notebooks anyway.
- **Decision: we keep the preprocessor as it is.** The gains are within the noise, while both changes have a cost. The ratio function would have to be pickled with the model, so the API would need our own Python code to load it. `IterativeImputer` is experimental and much harder to explain than a median. If an idea only helps within the noise, we choose the simpler version.
- The imputation question comes back in `05b_model_gradient_boosting.ipynb`: that model can handle missing values itself, without any imputation, which is an independent check of this conclusion.

## 10. Summary & open questions for the model notebooks

### What was done (EDA decision table → implementation)
| # | EDA step | Done here |
|---|---|---|
| 1 | snake_case column names | section 2 |
| 2 | Letter codes → readable labels | section 2 (with a check for unmapped codes) |
| 3 | Drop `jumbled_noise_0/1` | section 3, benefit confirmed in section 9 |
| 4 | Keep all rows | 5000 in, 5000 out |
| 5 | Categorical NaN → `"missing"` | section 4 |
| 6 | Numerical NaN → median, fitted on train only | section 7 |
| 7 | Keep outliers & zero stems, add `has_stem` | section 5, extended with `stem_surface = none` and logical filling of missing stem measures |
| 8 | Group rare categories | section 7, threshold 10 (not 30, see section 7) |
| 9 | `log1p` + scaling | section 7, applied for all models since it is harmless for trees |
| 10 | Stratified split | section 6: train / validation / test = 70 / 15 / 15 (3500 / 750 / 750), one folder per set |

### Output files
| Folder | Cleaned (readable values, numerical `NaN` kept) | Prepared (model-ready, 64 columns + `is_poisonous`) |
|---|---|---|
| `Data/train/` | `mushroom_cleaned_train.csv` | `mushroom_prepared_train.csv` |
| `Data/validation/` | `mushroom_cleaned_validation.csv` | `mushroom_prepared_validation.csv` |
| `Data/test/` | `mushroom_cleaned_test.csv` | `mushroom_prepared_test.csv` |

All files have the key `row_id` (the row number in the raw file). Plus `models/mushroom_preprocessor.joblib`, fitted on the training set.

### How to use the output in the model notebooks
```python
def data_path(split, kind):  # split: train / validation / test, kind: cleaned / prepared
    return Path("Data") / split / f"mushroom_{kind}_{split}.csv"

# Our own scikit-learn models: fully prepared, numeric data
train = pd.read_csv(data_path("train", "prepared"), index_col="row_id")
X_train, y_train = train.drop(columns="is_poisonous"), train["is_poisonous"]

# PyCaret / AutoML, native missing values, deploy pipelines: the cleaned data
train_df = pd.read_csv(data_path("train", "cleaned"), index_col="row_id")
```
The role of each set: **train** to fit and to tune with cross-validation, **validation** for the decision threshold and the choice between models, **test** once per model for the final numbers.

For the API: apply the same cleaning (sections 2–5) to the user input, then `joblib.load("models/mushroom_preprocessor.joblib").transform(...)`.

### Open questions
Answered in section 9.2:
- *Drop `spore_print_color`?* No: dropping it lowers the AUC in most folds. Kept as is, with `"missing"`.
- *Median vs. iterative imputation* and *ratio features*: small gains within the noise. Not adopted; median imputation stays.

Left for the model notebooks:
- Remove or relabel suspected flipped labels (EDA section 9): **training set only**, and never remove *poisonous* labels just because a model disagrees.
- Decision threshold and class weights to raise recall on poisonous mushrooms (threshold on the validation set).